# 04. Commercial bank selection

Match bank records with historical FDIC classifications and retain commercial banks. Review unmatched records and identifier changes before proceeding.


## 1. Mount Drive, import configuration and lock the scope


In [ ]:
# Mount Drive, import main analysis configuration and lock the scope

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    import pyarrow  # noqa: F401
except ImportError:
    print("Installing the missing pyarrow Parquet dependency...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
    import pyarrow  # noqa: F401

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable

PROJECT_ROOT = Path("/content/drive/MyDrive/Data Final")
CONFIG_PATH = PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared V2 config not found: {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config)

assert config.ANALYSIS_VERSION == "v2_bank_lag1_main"
assert config.ARCHIVED_V1_READ_ONLY is True

V2_ROOT = config.V2_ROOT
V2_OUTPUT_ROOT = config.V2_OUTPUT_ROOT
V2_PROCESSED_DATA_DIR = config.V2_PROCESSED_DATA_DIR
FDIC_FILE = config.FDIC_FILE

NB03_OUTPUT_DIR = V2_OUTPUT_ROOT / "03_FFIEC_Data_Extraction_and_Harmonisation"
NB04_OUTPUT_DIR = V2_OUTPUT_ROOT / "04_Commercial_Bank_Filter"
FDIC_CLASSIFICATION_RAW_DIR = NB04_OUTPUT_DIR / "Historical_FDIC_Classifications"
FFIEC_INPUT_PATH = V2_PROCESSED_DATA_DIR / "harmonised_ffiec_panel_v2.parquet"
PANEL_OUTPUT_PATH = V2_PROCESSED_DATA_DIR / "commercial_bank_ffiec_panel_v2.parquet"

NB04_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FDIC_CLASSIFICATION_RAW_DIR.mkdir(parents=True, exist_ok=True)

ELIGIBLE_COMMERCIAL_CLASSES = ("N", "NM", "SM")
KNOWN_INELIGIBLE_CLASSES = ("SB", "SI", "SL", "OI", "NC")
EXPECTED_CLASSES = ELIGIBLE_COMMERCIAL_CLASSES + KNOWN_INELIGIBLE_CLASSES

institution_class_dictionary = pd.DataFrame({
    "institution_class": ["N", "NM", "SM", "SB", "SI", "SL", "OI", "NC"],
    "institution_type": [
        "National commercial bank", "State non-member commercial bank",
        "State member commercial bank", "State savings bank",
        "State savings institution", "Federal savings association",
        "Insured U.S. branch of a foreign institution", "Non-deposit trust company",
    ],
    "commercial_bank_eligible": [True, True, True, False, False, False, False, False],
})

setup_gate = pd.DataFrame({
    "check": [
        "V2 config loaded", "Notebook 03 panel exists", "FDIC failure file exists",
        "Eligible classes are exactly N, NM and SM", "V1 is read only",
    ],
    "passed": [
        config.ANALYSIS_VERSION == "v2_bank_lag1_main", FFIEC_INPUT_PATH.is_file(),
        FDIC_FILE.is_file(), set(ELIGIBLE_COMMERCIAL_CLASSES) == {"N", "NM", "SM"},
        config.ARCHIVED_V1_READ_ONLY is True,
    ],
})
setup_gate["status"] = setup_gate["passed"].map({True: "PASS", False: "FAIL"})
display(setup_gate)
if not setup_gate["passed"].all():
    raise AssertionError("Notebook 04 setup gate failed.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)
print(f"Input panel: {FFIEC_INPUT_PATH}")
print(f"Audit folder: {NB04_OUTPUT_DIR}")


## 2. Load and independently validate the Notebook 03 panel


In [ ]:
# Load and independently validate the Notebook 03 panel

ffiec_panel = pd.read_parquet(FFIEC_INPUT_PATH)
required_columns = {
    "report_date", "idrssd", "cert", "bank_name", "filing_form",
    "total_assets", "total_loans", "total_deposits", "total_equity_capital",
}
missing_columns = sorted(required_columns - set(ffiec_panel.columns))
if missing_columns:
    raise KeyError(f"Notebook 03 panel is missing required columns: {missing_columns}")

ffiec_panel["report_date"] = pd.to_datetime(ffiec_panel["report_date"], errors="raise").dt.normalize()
ffiec_panel["idrssd"] = pd.to_numeric(ffiec_panel["idrssd"], errors="raise").astype("int64")
ffiec_panel["cert"] = pd.to_numeric(ffiec_panel["cert"], errors="raise").astype("int64")
ffiec_panel["bank_name"] = ffiec_panel["bank_name"].astype("string").str.strip()

expected_quarters = pd.period_range("2001Q1", "2026Q1", freq="Q").to_timestamp(how="end").normalize()
observed_quarters = pd.DatetimeIndex(sorted(ffiec_panel["report_date"].unique()))
duplicate_idrssd_quarter_input = ffiec_panel.loc[
    ffiec_panel.duplicated(["idrssd", "report_date"], keep=False)
].sort_values(["report_date", "idrssd"])
duplicate_cert_quarter_input = ffiec_panel.loc[
    ffiec_panel["cert"].gt(0) & ffiec_panel.duplicated(["cert", "report_date"], keep=False)
].sort_values(["report_date", "cert"])

zero_cert_rows = ffiec_panel.loc[
    ffiec_panel["cert"].le(0),
    ["report_date", "idrssd", "cert", "bank_name", "filing_form"],
].copy()

input_gate = pd.DataFrame({
    "check": [
        "Input is nonempty", "Dates are exact quarter ends", "Complete 101-quarter grid",
        "IDRSSD is positive", "CERT is nonnegative", "Total assets is positive",
        "No duplicate IDRSSD-quarter rows", "No duplicate positive-CERT-quarter rows",
    ],
    "passed": [
        len(ffiec_panel) > 0,
        ffiec_panel["report_date"].eq(ffiec_panel["report_date"].dt.to_period("Q").dt.end_time.dt.normalize()).all(),
        observed_quarters.equals(expected_quarters), ffiec_panel["idrssd"].gt(0).all(),
        ffiec_panel["cert"].ge(0).all(), ffiec_panel["total_assets"].gt(0).all(),
        duplicate_idrssd_quarter_input.empty, duplicate_cert_quarter_input.empty,
    ],
})
input_gate["status"] = input_gate["passed"].map({True: "PASS", False: "FAIL"})
display(input_gate)
print(f"Input observations: {len(ffiec_panel):,}")
print(f"Unique IDRSSD: {ffiec_panel['idrssd'].nunique():,}")
print(f"Unique positive CERT: {ffiec_panel.loc[ffiec_panel['cert'].gt(0), 'cert'].nunique():,}")
print(f"Zero-CERT observations requiring IDRSSD recovery: {len(zero_cert_rows):,}")
display(zero_cert_rows)
if not input_gate["passed"].all():
    raise AssertionError("Notebook 03 input validation failed.")


## 3. Define the reproducible FDIC historical-classification download


In [ ]:
# Define the reproducible FDIC historical-classification download

FDIC_API_URL = "https://api.fdic.gov/banks/financials"
FDIC_FIELDS = ["REPDTE", "CERT", "RSSDID", "NAME", "BKCLASS"]
FDIC_PAGE_LIMIT = 10000

retry_strategy = Retry(
    total=5, connect=5, read=5, status=5, backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]), respect_retry_after_header=True,
)
fdic_session = requests.Session()
fdic_session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

def download_fdic_quarter(report_date, save_raw=True):
    report_date = pd.Timestamp(report_date).normalize()
    repdte = report_date.strftime("%Y%m%d")
    records, page_audit, offset, expected_total = [], [], 0, None
    first_payload = None
    while True:
        params = {
            "filters": f"REPDTE:{repdte}", "fields": ",".join(FDIC_FIELDS),
            "sort_by": "CERT", "sort_order": "ASC", "limit": FDIC_PAGE_LIMIT,
            "offset": offset, "format": "json",
        }
        response = fdic_session.get(FDIC_API_URL, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()
        if first_payload is None:
            first_payload = payload
        items = payload.get("data", [])
        if expected_total is None:
            expected_total = int(payload.get("meta", {}).get("total", payload.get("totals", {}).get("count", 0)))
        records.extend(items)
        page_audit.append({"report_date": report_date, "offset": offset, "records": len(items), "http_status": response.status_code})
        if len(records) >= expected_total:
            break
        if not items:
            raise RuntimeError(f"FDIC pagination stopped early for {report_date.date()}.")
        offset += FDIC_PAGE_LIMIT
        time.sleep(1.1)

    if len(records) != expected_total:
        raise AssertionError(f"FDIC total mismatch for {report_date.date()}.")
    extracted = [item.get("data", item) if isinstance(item, dict) else item for item in records]
    raw = pd.DataFrame(extracted)
    missing = [field for field in FDIC_FIELDS if field not in raw]
    if missing:
        raise KeyError(f"FDIC response missing fields: {missing}")
    result = raw[FDIC_FIELDS].rename(columns={
        "REPDTE": "report_date", "CERT": "cert", "RSSDID": "idrssd",
        "NAME": "bank_name_fdic", "BKCLASS": "institution_class",
    })
    result["report_date"] = pd.to_datetime(result["report_date"].astype("string"), format="%Y%m%d", errors="raise")
    for column in ["cert", "idrssd"]:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype("Int64")
    result["bank_name_fdic"] = result["bank_name_fdic"].astype("string").str.strip()
    result["institution_class"] = result["institution_class"].astype("string").str.strip().str.upper()
    result = result.sort_values(["report_date", "cert", "idrssd"], kind="mergesort").reset_index(drop=True)

    raw_path = FDIC_CLASSIFICATION_RAW_DIR / f"fdic_classification_{repdte}_raw.json"
    if save_raw:
        archive = {
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "endpoint": FDIC_API_URL, "report_date": str(report_date.date()),
            "fields": FDIC_FIELDS, "expected_total": expected_total,
            "page_audit": page_audit, "meta": first_payload.get("meta", {}), "data": records,
        }
        raw_path.write_text(json.dumps(archive, indent=2, default=str), encoding="utf-8")
    return result, {
        "report_date": report_date, "records_downloaded": len(result),
        "pages_downloaded": len(page_audit), "raw_output_path": str(raw_path),
    }

# A live smoke test validates the current API schema without hard-coding current counts.
fdic_test, fdic_test_audit = download_fdic_quarter(expected_quarters[-1], save_raw=True)
test_classes = set(fdic_test["institution_class"].dropna())
api_test_gate = pd.DataFrame({
    "check": ["Latest quarter is nonempty", "Required fields standardised", "Classes recognised", "No duplicate CERT-quarter", "No duplicate IDRSSD-quarter"],
    "passed": [
        len(fdic_test) > 0,
        set(["report_date", "cert", "idrssd", "bank_name_fdic", "institution_class"]).issubset(fdic_test.columns),
        test_classes.issubset(set(EXPECTED_CLASSES)),
        not fdic_test.loc[fdic_test["cert"].gt(0)].duplicated(["report_date", "cert"]).any(),
        not fdic_test.loc[fdic_test["idrssd"].gt(0)].duplicated(["report_date", "idrssd"]).any(),
    ],
})
api_test_gate["status"] = api_test_gate["passed"].map({True: "PASS", False: "FAIL"})
display(api_test_gate)
display(fdic_test["institution_class"].value_counts(dropna=False).rename_axis("class").reset_index(name="institutions"))
if not api_test_gate["passed"].all():
    raise AssertionError("FDIC API smoke test failed.")


## 4. Download all 101 historical FDIC classification quarters


In [ ]:
# Download all 101 historical FDIC classification quarters

quarter_tables, request_audits = [], []
for report_date in tqdm(expected_quarters, desc="FDIC historical classifications"):
    cached_csv = FDIC_CLASSIFICATION_RAW_DIR / f"fdic_classification_{report_date:%Y%m%d}_standardised.csv"
    if cached_csv.is_file():
        quarter = pd.read_csv(cached_csv, parse_dates=["report_date"], dtype={"institution_class": "string"})
        quarter["cert"] = pd.to_numeric(quarter["cert"], errors="coerce").astype("Int64")
        quarter["idrssd"] = pd.to_numeric(quarter["idrssd"], errors="coerce").astype("Int64")
        audit = {"report_date": report_date, "records_downloaded": len(quarter), "pages_downloaded": pd.NA, "source": "cached_standardised_csv"}
    else:
        quarter, audit = download_fdic_quarter(report_date, save_raw=True)
        quarter.to_csv(cached_csv, index=False)
        audit["source"] = "live_api"
        time.sleep(1.1)
    quarter_tables.append(quarter)
    request_audits.append(audit)

fdic_history = pd.concat(quarter_tables, ignore_index=True)
fdic_request_audit = pd.DataFrame(request_audits)
fdic_history["report_date"] = pd.to_datetime(fdic_history["report_date"], errors="raise").dt.normalize()
for column in ["cert", "idrssd"]:
    fdic_history[column] = pd.to_numeric(fdic_history[column], errors="coerce").astype("Int64")

unknown_classes = sorted(set(fdic_history["institution_class"].dropna()) - set(EXPECTED_CLASSES))
fdic_duplicate_cert_quarter = fdic_history.loc[
    fdic_history["cert"].gt(0) & fdic_history.duplicated(["report_date", "cert"], keep=False)
].sort_values(["report_date", "cert"])
fdic_duplicate_idrssd_quarter = fdic_history.loc[
    fdic_history["idrssd"].gt(0) & fdic_history.duplicated(["report_date", "idrssd"], keep=False)
].sort_values(["report_date", "idrssd"])

historical_gate = pd.DataFrame({
    "check": [
        "All 101 quarters downloaded", "FDIC quarter coverage is complete", "No unknown institution classes",
        "No duplicate positive CERT-quarter", "No duplicate positive IDRSSD-quarter",
    ],
    "passed": [
        len(fdic_request_audit) == len(expected_quarters),
        pd.DatetimeIndex(sorted(fdic_history["report_date"].unique())).equals(expected_quarters),
        len(unknown_classes) == 0, fdic_duplicate_cert_quarter.empty, fdic_duplicate_idrssd_quarter.empty,
    ],
})
historical_gate["status"] = historical_gate["passed"].map({True: "PASS", False: "FAIL"})
display(historical_gate)
print(f"Historical FDIC rows: {len(fdic_history):,}")
print(f"Reporting quarters: {fdic_history['report_date'].nunique():,}")
display(fdic_history["institution_class"].value_counts().rename_axis("class").reset_index(name="observations"))
if not historical_gate["passed"].all():
    raise AssertionError("Historical FDIC classification gate failed.")


## 5. Construct the historical CERT–IDRSSD crosswalk and exact-quarter matches


In [ ]:
# Construct the historical CERT–IDRSSD crosswalk and exact-quarter matches

crosswalk = fdic_history.loc[
    fdic_history["cert"].gt(0) & fdic_history["idrssd"].gt(0),
    ["cert", "idrssd", "report_date", "bank_name_fdic", "institution_class"],
].copy()
crosswalk_pair_summary = (
    crosswalk.groupby(["cert", "idrssd"], as_index=False)
    .agg(first_report_date=("report_date", "min"), last_report_date=("report_date", "max"),
         observations=("report_date", "size"), institution_classes=("institution_class", lambda s: "|".join(sorted(set(s.dropna())))))
)
cert_to_multiple_idrssd = (
    crosswalk_pair_summary.groupby("cert")["idrssd"].nunique().loc[lambda s: s.gt(1)].rename("idrssd_count").reset_index()
)
idrssd_to_multiple_cert = (
    crosswalk_pair_summary.groupby("idrssd")["cert"].nunique().loc[lambda s: s.gt(1)].rename("cert_count").reset_index()
)

ffiec = ffiec_panel.reset_index(drop=False).rename(columns={"index": "ffiec_row_id"})
fdic_cert = fdic_history.loc[fdic_history["cert"].gt(0)].rename(columns={
    "idrssd": "fdic_idrssd_by_cert", "bank_name_fdic": "fdic_name_by_cert", "institution_class": "class_by_cert"
})[["report_date", "cert", "fdic_idrssd_by_cert", "fdic_name_by_cert", "class_by_cert"]]
fdic_rssd = fdic_history.loc[fdic_history["idrssd"].gt(0)].rename(columns={
    "cert": "fdic_cert_by_idrssd", "bank_name_fdic": "fdic_name_by_idrssd", "institution_class": "class_by_idrssd"
})[["report_date", "idrssd", "fdic_cert_by_idrssd", "fdic_name_by_idrssd", "class_by_idrssd"]]

matched = ffiec.merge(fdic_cert, on=["report_date", "cert"], how="left", validate="many_to_one")
matched = matched.merge(fdic_rssd, on=["report_date", "idrssd"], how="left", validate="many_to_one")
matched["cert_match"] = matched["class_by_cert"].notna()
matched["idrssd_match"] = matched["class_by_idrssd"].notna()
matched["dual_match"] = matched["cert_match"] & matched["idrssd_match"]
matched["class_conflict"] = matched["dual_match"] & matched["class_by_cert"].ne(matched["class_by_idrssd"])
matched["identifier_conflict"] = matched["dual_match"] & (
    matched["fdic_idrssd_by_cert"].ne(matched["idrssd"]) | matched["fdic_cert_by_idrssd"].ne(matched["cert"])
)
matched["institution_class"] = matched["class_by_cert"].combine_first(matched["class_by_idrssd"])
matched["matching_route"] = np.select(
    [matched["class_conflict"] | matched["identifier_conflict"], matched["dual_match"], matched["cert_match"], matched["idrssd_match"]],
    ["CONFLICT", "CERT_AND_IDRSSD", "CERT_ONLY", "IDRSSD_ONLY"], default="UNMATCHED",
)
conflicts = matched.loc[matched["matching_route"].eq("CONFLICT")].copy()
unmatched = matched.loc[matched["matching_route"].eq("UNMATCHED")].copy()
zero_cert_recovery = matched.loc[matched["cert"].le(0), [
    "ffiec_row_id", "report_date", "idrssd", "cert", "bank_name", "matching_route",
    "fdic_cert_by_idrssd", "fdic_name_by_idrssd", "class_by_idrssd",
]].copy()

exact_match_rate = matched["institution_class"].notna().mean()
matching_summary = pd.DataFrame({
    "measure": [
        "FFIEC observations", "Matched observations", "Unmatched observations", "Conflicting observations",
        "Match rate percent", "CERT and IDRSSD matches", "CERT-only matches", "IDRSSD-only matches",
        "Historical CERT-IDRSSD pairs", "CERT values linked to multiple IDRSSD", "IDRSSD values linked to multiple CERT",
    ],
    "value": [
        len(matched), matched["institution_class"].notna().sum(), len(unmatched), len(conflicts),
        exact_match_rate * 100, matched["matching_route"].eq("CERT_AND_IDRSSD").sum(),
        matched["matching_route"].eq("CERT_ONLY").sum(), matched["matching_route"].eq("IDRSSD_ONLY").sum(),
        len(crosswalk_pair_summary), len(cert_to_multiple_idrssd), len(idrssd_to_multiple_cert),
    ],
})
display(matching_summary)
print("Conflict rows:")
display(conflicts[["report_date", "idrssd", "cert", "bank_name", "class_by_cert", "class_by_idrssd", "fdic_idrssd_by_cert", "fdic_cert_by_idrssd"]])
print("Zero-CERT recovery audit:")
display(zero_cert_recovery)


## 6. Investigate unmatched identifiers without imputing classifications


In [ ]:
# Investigate unmatched identifiers without imputing classifications

def normalise_name(series):
    return (series.fillna("").astype(str).str.upper().str.replace(r"[^A-Z0-9]+", " ", regex=True).str.strip().str.replace(r"\s+", " ", regex=True))

fdic_audit = fdic_history.copy()
fdic_audit["normalised_name"] = normalise_name(fdic_audit["bank_name_fdic"])
unmatched_audit_rows, neighbouring_rows, name_candidate_rows = [], [], []

for row in unmatched.itertuples(index=False):
    cert_history = fdic_audit.loc[fdic_audit["cert"].eq(row.cert)] if row.cert > 0 else fdic_audit.iloc[0:0]
    rssd_history = fdic_audit.loc[fdic_audit["idrssd"].eq(row.idrssd)]
    same_quarter = fdic_audit.loc[fdic_audit["report_date"].eq(row.report_date)]
    norm_name = normalise_name(pd.Series([row.bank_name])).iloc[0]
    name_candidates = same_quarter.loc[same_quarter["normalised_name"].eq(norm_name)] if norm_name else same_quarter.iloc[0:0]

    for candidate in name_candidates.itertuples(index=False):
        name_candidate_rows.append({
            "ffiec_row_id": row.ffiec_row_id, "report_date": row.report_date, "ffiec_idrssd": row.idrssd,
            "ffiec_cert": row.cert, "ffiec_bank_name": row.bank_name, "candidate_idrssd": candidate.idrssd,
            "candidate_cert": candidate.cert, "candidate_name": candidate.bank_name_fdic,
            "candidate_class": candidate.institution_class, "evidence_use": "AUDIT_ONLY_NOT_IMPUTED",
        })

    context = {}
    for route, history in [("cert", cert_history), ("idrssd", rssd_history)]:
        for direction in ["previous", "next"]:
            candidates = history.loc[history["report_date"].lt(row.report_date)] if direction == "previous" else history.loc[history["report_date"].gt(row.report_date)]
            candidates = candidates.sort_values("report_date", ascending=(direction == "next"))
            nearest = None if candidates.empty else candidates.iloc[0]
            context[f"{route}_{direction}_date"] = pd.NaT if nearest is None else nearest["report_date"]
            context[f"{route}_{direction}_class"] = pd.NA if nearest is None else nearest["institution_class"]
            neighbouring_rows.append({
                "ffiec_row_id": row.ffiec_row_id, "report_date": row.report_date, "idrssd": row.idrssd, "cert": row.cert,
                "audit_route": route, "direction": direction, "context_found": nearest is not None,
                "context_report_date": pd.NaT if nearest is None else nearest["report_date"],
                "context_idrssd": pd.NA if nearest is None else nearest["idrssd"],
                "context_cert": pd.NA if nearest is None else nearest["cert"],
                "context_class": pd.NA if nearest is None else nearest["institution_class"],
                "evidence_use": "AUDIT_ONLY_NOT_IMPUTED",
            })

    unmatched_audit_rows.append({
        "ffiec_row_id": row.ffiec_row_id, "report_date": row.report_date, "idrssd": row.idrssd, "cert": row.cert,
        "bank_name": row.bank_name, "filing_form": row.filing_form,
        "cert_history_records": len(cert_history), "idrssd_history_records": len(rssd_history),
        "exact_quarter_name_candidates": len(name_candidates), **context,
        "final_resolution": "EXCLUDE_NO_EXACT_QUARTER_EVIDENCE",
        "resolution_basis": "No exact CERT-quarter or IDRSSD-quarter classification; neighbour and name evidence not imputed.",
    })

unmatched_investigation = pd.DataFrame(unmatched_audit_rows)
neighbouring_context = pd.DataFrame(neighbouring_rows)
exact_name_candidates = pd.DataFrame(name_candidate_rows)
display(unmatched_investigation)
if not exact_name_candidates.empty:
    print("Exact-quarter name candidates (audit only):")
    display(exact_name_candidates)


## 7. Audit failed-bank identifier matching and eligibility


In [ ]:
# Audit failed-bank identifier matching and eligibility

failed_raw = pd.read_excel(FDIC_FILE)
normalised_failure_columns = {re.sub(r"[^a-z0-9]+", "", str(c).lower()): c for c in failed_raw.columns}

def find_failure_column(patterns, required=True):
    for pattern in patterns:
        for normalised, original in normalised_failure_columns.items():
            if pattern in normalised:
                return original
    if required:
        raise KeyError(f"Could not identify failure-file column from patterns: {patterns}")
    return None

failure_cert_column = find_failure_column(["cert", "certificate"])
failure_date_column = find_failure_column(["faildate", "failuredate", "closingdate", "dateclosed"])
failure_name_column = find_failure_column(["bankname", "institutionname", "name"], required=False)

failures = pd.DataFrame({
    "cert": pd.to_numeric(failed_raw[failure_cert_column], errors="coerce").astype("Int64"),
    "failure_date": pd.to_datetime(failed_raw[failure_date_column], errors="coerce"),
    "failed_bank_name": failed_raw[failure_name_column].astype("string").str.strip() if failure_name_column else pd.Series(pd.NA, index=failed_raw.index, dtype="string"),
}).dropna(subset=["cert", "failure_date"])
failures = failures.loc[failures["cert"].gt(0)].drop_duplicates(["cert", "failure_date"]).reset_index(drop=True)

failure_rows = []
for failure in failures.itertuples(index=False):
    cert_records = fdic_history.loc[fdic_history["cert"].eq(failure.cert)].sort_values("report_date")
    ffiec_records = matched.loc[matched["cert"].eq(failure.cert)].sort_values("report_date")
    pre_failure = ffiec_records.loc[ffiec_records["report_date"].lt(failure.failure_date)]
    last_pre = None if pre_failure.empty else pre_failure.iloc[-1]
    failure_rows.append({
        "cert": int(failure.cert), "failure_date": failure.failure_date, "failed_bank_name": failure.failed_bank_name,
        "fdic_history_found": not cert_records.empty, "ffiec_history_found": not ffiec_records.empty,
        "ffiec_pre_failure_history_found": not pre_failure.empty,
        "last_pre_failure_report_date": pd.NaT if last_pre is None else last_pre["report_date"],
        "last_pre_failure_idrssd": pd.NA if last_pre is None else last_pre["idrssd"],
        "last_pre_failure_class": pd.NA if last_pre is None else last_pre["institution_class"],
        "last_pre_failure_match_route": pd.NA if last_pre is None else last_pre["matching_route"],
        "commercial_bank_eligible_at_last_pre_failure_filing": False if last_pre is None else last_pre["institution_class"] in ELIGIBLE_COMMERCIAL_CLASSES,
    })
failed_bank_audit = pd.DataFrame(failure_rows)
failed_total = len(failed_bank_audit)
failed_ffiec_matched = int(failed_bank_audit["ffiec_pre_failure_history_found"].sum()) if failed_total else 0
failed_eligible = int(failed_bank_audit["commercial_bank_eligible_at_last_pre_failure_filing"].sum()) if failed_total else 0
failed_bank_summary = pd.DataFrame({
    "measure": [
        "Usable FDIC failure records", "Failures with pre-failure FFIEC history", "Failed-bank matching rate percent",
        "Commercial-bank failures at last pre-failure filing", "Commercial-bank eligibility rate percent",
    ],
    "value": [
        failed_total, failed_ffiec_matched, 100 * failed_ffiec_matched / failed_total if failed_total else np.nan,
        failed_eligible, 100 * failed_eligible / failed_total if failed_total else np.nan,
    ],
})
display(failed_bank_summary)
print("Failed banks without pre-failure FFIEC history:")
display(failed_bank_audit.loc[~failed_bank_audit["ffiec_pre_failure_history_found"]])
print("Failed banks not eligible as commercial banks at their last pre-failure filing:")
display(failed_bank_audit.loc[failed_bank_audit["ffiec_pre_failure_history_found"] & ~failed_bank_audit["commercial_bank_eligible_at_last_pre_failure_filing"]])


## 8. Apply the commercial-bank restriction and validate the final panel


In [ ]:
# Apply the commercial-bank restriction and validate the final panel

if not conflicts.empty:
    raise AssertionError("Conflicting exact-quarter identifiers remain. Review conflicts before filtering.")

matched["commercial_bank_eligible"] = matched["institution_class"].isin(ELIGIBLE_COMMERCIAL_CLASSES)
matched["filter_decision"] = np.select(
    [matched["matching_route"].eq("UNMATCHED"), matched["commercial_bank_eligible"]],
    ["EXCLUDE_NO_EXACT_QUARTER_EVIDENCE", "RETAIN_COMMERCIAL_BANK"],
    default="EXCLUDE_NONCOMMERCIAL_CLASS",
)
matched["filter_reason"] = np.select(
    [matched["matching_route"].eq("UNMATCHED"), matched["commercial_bank_eligible"]],
    ["No exact CERT-quarter or IDRSSD-quarter FDIC classification", "Exact-quarter FDIC class is N, NM or SM"],
    default="Exact-quarter FDIC class is not N, NM or SM",
)

source_columns = list(ffiec_panel.columns)
final_columns = source_columns + ["institution_class", "matching_route", "commercial_bank_eligible"]
commercial_bank_panel = matched.loc[matched["commercial_bank_eligible"], final_columns].copy()
commercial_bank_panel = commercial_bank_panel.sort_values(["report_date", "idrssd"], kind="mergesort").reset_index(drop=True)
excluded_observations = matched.loc[~matched["commercial_bank_eligible"], source_columns + ["institution_class", "matching_route", "filter_decision", "filter_reason"]].copy()

duplicate_idrssd_quarter_final = commercial_bank_panel.loc[
    commercial_bank_panel.duplicated(["idrssd", "report_date"], keep=False)
]
duplicate_cert_quarter_final = commercial_bank_panel.loc[
    commercial_bank_panel["cert"].gt(0) & commercial_bank_panel.duplicated(["cert", "report_date"], keep=False)
]
retained_class_distribution = commercial_bank_panel["institution_class"].value_counts().reindex(ELIGIBLE_COMMERCIAL_CLASSES, fill_value=0).rename_axis("institution_class").reset_index(name="observations")
excluded_class_distribution = excluded_observations["institution_class"].fillna("UNMATCHED").value_counts().rename_axis("institution_class").reset_index(name="observations")
quarterly_filter_summary = matched.groupby("report_date", as_index=False).agg(
    input_observations=("ffiec_row_id", "size"),
    matched_observations=("institution_class", "count"),
    retained_observations=("commercial_bank_eligible", "sum"),
)
quarterly_filter_summary["excluded_observations"] = quarterly_filter_summary["input_observations"] - quarterly_filter_summary["retained_observations"]
quarterly_filter_summary["matching_rate"] = quarterly_filter_summary["matched_observations"] / quarterly_filter_summary["input_observations"]

final_gate = pd.DataFrame({
    "check": [
        "Every input row has one decision", "Only N, NM and SM retained", "No missing retained class",
        "Retained plus excluded reconciles", "No duplicate IDRSSD-quarter", "No duplicate positive-CERT-quarter",
        "All retained matches are exact-quarter", "Source values and column order preserved",
    ],
    "passed": [
        matched["filter_decision"].notna().all(),
        set(commercial_bank_panel["institution_class"].unique()).issubset(set(ELIGIBLE_COMMERCIAL_CLASSES)),
        commercial_bank_panel["institution_class"].notna().all(),
        len(commercial_bank_panel) + len(excluded_observations) == len(ffiec_panel),
        duplicate_idrssd_quarter_final.empty, duplicate_cert_quarter_final.empty,
        commercial_bank_panel["matching_route"].isin(["CERT_AND_IDRSSD", "CERT_ONLY", "IDRSSD_ONLY"]).all(),
        list(commercial_bank_panel.columns[:len(source_columns)]) == source_columns,
    ],
})
final_gate["status"] = final_gate["passed"].map({True: "PASS", False: "FAIL"})
display(final_gate)
display(retained_class_distribution)
display(excluded_class_distribution)
if not final_gate["passed"].all():
    raise AssertionError("Commercial-bank filtering gate failed.")


## 9. Save the authoritative panel, audit tables and final report


In [ ]:
# Save the authoritative panel, audit tables and final report

output_tables = {
    "institution_class_dictionary_v2.csv": institution_class_dictionary,
    "input_validation_gate_v2.csv": input_gate,
    "zero_cert_recovery_v2.csv": zero_cert_recovery,
    "fdic_request_audit_v2.csv": fdic_request_audit,
    "fdic_duplicate_cert_quarter_v2.csv": fdic_duplicate_cert_quarter,
    "fdic_duplicate_idrssd_quarter_v2.csv": fdic_duplicate_idrssd_quarter,
    "cert_idrssd_crosswalk_v2.csv": crosswalk_pair_summary,
    "cert_to_multiple_idrssd_v2.csv": cert_to_multiple_idrssd,
    "idrssd_to_multiple_cert_v2.csv": idrssd_to_multiple_cert,
    "matching_summary_v2.csv": matching_summary,
    "identifier_conflicts_v2.csv": conflicts,
    "unmatched_investigation_v2.csv": unmatched_investigation,
    "neighbouring_quarter_context_v2.csv": neighbouring_context,
    "exact_name_candidates_v2.csv": exact_name_candidates,
    "failed_bank_matching_audit_v2.csv": failed_bank_audit,
    "failed_bank_matching_summary_v2.csv": failed_bank_summary,
    "filter_decisions_v2.csv": matched[["ffiec_row_id", "report_date", "idrssd", "cert", "bank_name", "institution_class", "matching_route", "commercial_bank_eligible", "filter_decision", "filter_reason"]],
    "excluded_observations_v2.csv": excluded_observations,
    "retained_class_distribution_v2.csv": retained_class_distribution,
    "excluded_class_distribution_v2.csv": excluded_class_distribution,
    "quarterly_filter_summary_v2.csv": quarterly_filter_summary,
    "duplicate_idrssd_quarter_final_v2.csv": duplicate_idrssd_quarter_final,
    "duplicate_cert_quarter_final_v2.csv": duplicate_cert_quarter_final,
    "final_validation_gate_v2.csv": final_gate,
}
for filename, table in output_tables.items():
    table.to_csv(NB04_OUTPUT_DIR / filename, index=False)

temp_panel_path = PANEL_OUTPUT_PATH.with_suffix(".tmp.parquet")
commercial_bank_panel.to_parquet(temp_panel_path, index=False)
temp_panel_path.replace(PANEL_OUTPUT_PATH)
saved_panel = pd.read_parquet(PANEL_OUTPUT_PATH)

panel_reloaded_exactly = (
    saved_panel.shape == commercial_bank_panel.shape
    and list(saved_panel.columns) == list(commercial_bank_panel.columns)
    and pd.util.hash_pandas_object(saved_panel, index=True).sum() == pd.util.hash_pandas_object(commercial_bank_panel, index=True).sum()
)
if not panel_reloaded_exactly:
    raise AssertionError("Saved Parquet panel did not reload exactly.")

warning_messages = []
if len(unmatched): warning_messages.append(f"{len(unmatched)} unmatched observations excluded")
if failed_total and failed_ffiec_matched < failed_total: warning_messages.append(f"{failed_total - failed_ffiec_matched} FDIC failure records have no pre-failure FFIEC history")

final_result = {
    "overall_status": "PASS",
    "input_observations": int(len(ffiec_panel)),
    "matched_observations": int(matched["institution_class"].notna().sum()),
    "overall_match_rate_percent": float(100 * matched["institution_class"].notna().mean()),
    "commercial_bank_observations_retained": int(len(commercial_bank_panel)),
    "observations_excluded": int(len(excluded_observations)),
    "unique_retained_idrssd": int(commercial_bank_panel["idrssd"].nunique()),
    "unique_retained_cert": int(commercial_bank_panel.loc[commercial_bank_panel["cert"].gt(0), "cert"].nunique()),
    "reporting_quarters": int(commercial_bank_panel["report_date"].nunique()),
    "unmatched_observations_excluded": int(len(unmatched)),
    "identifier_conflicts": int(len(conflicts)),
    "duplicate_idrssd_quarter": int(len(duplicate_idrssd_quarter_final)),
    "duplicate_cert_quarter": int(len(duplicate_cert_quarter_final)),
    "failed_bank_records": int(failed_total),
    "failed_banks_with_pre_failure_ffiec_history": int(failed_ffiec_matched),
    "failed_bank_matching_rate_percent": float(100 * failed_ffiec_matched / failed_total) if failed_total else None,
    "panel_path": str(PANEL_OUTPUT_PATH),
    "audit_folder": str(NB04_OUTPUT_DIR),
    "warnings": warning_messages,
}
(NB04_OUTPUT_DIR / "audit_report_v2.json").write_text(json.dumps(final_result, indent=2, default=str), encoding="utf-8")

print("=" * 78)
print("NOTEBOOK 04 V2 — FINAL COMMERCIAL-BANK FILTER RESULT")
print("=" * 78)
print("Overall status: PASS")
print(f"FFIEC input observations: {len(ffiec_panel):,}")
print(f"Matched observations: {matched['institution_class'].notna().sum():,}")
print(f"Overall matching rate: {100 * matched['institution_class'].notna().mean():.6f}%")
print(f"Commercial-bank observations retained: {len(commercial_bank_panel):,}")
print(f"Observations excluded: {len(excluded_observations):,}")
print(f"Unique retained IDRSSD: {commercial_bank_panel['idrssd'].nunique():,}")
print(f"Unique retained CERT: {commercial_bank_panel.loc[commercial_bank_panel['cert'].gt(0), 'cert'].nunique():,}")
print(f"Reporting quarters: {commercial_bank_panel['report_date'].nunique():,}")
print(f"Unmatched observations excluded: {len(unmatched):,}")
print(f"Identifier conflicts: {len(conflicts):,}")
print(f"Historical CERT values linked to multiple IDRSSD: {len(cert_to_multiple_idrssd):,}")
print(f"Historical IDRSSD values linked to multiple CERT: {len(idrssd_to_multiple_cert):,}")
print(f"Duplicate IDRSSD-quarter rows: {len(duplicate_idrssd_quarter_final):,}")
print(f"Duplicate CERT-quarter rows: {len(duplicate_cert_quarter_final):,}")
print(f"Failed-bank records audited: {failed_total:,}")
print(f"Failed banks with pre-failure FFIEC history: {failed_ffiec_matched:,}")
if failed_total:
    print(f"Failed-bank matching rate: {100 * failed_ffiec_matched / failed_total:.6f}%")
print(f"Panel: {PANEL_OUTPUT_PATH}")
print(f"Audit folder: {NB04_OUTPUT_DIR}")
if warning_messages:
    print("\nNotebook 04 completed with warnings:")
    for warning in warning_messages: print(f"- {warning}")
    print("Review the final results and relevant warning audit rows before Notebook 05.")
else:
    print("\nNotebook 04 completed without warnings.")
print("Do not run Notebook 05 until this classification audit has been reviewed.")


In [ ]:
# Review unmatched and failed-bank records

print("Nine unmatched observations:")
display(unmatched_investigation)

print("Two failure records without pre-failure FFIEC history:")
display(
    failed_bank_audit.loc[
        ~failed_bank_audit["ffiec_pre_failure_history_found"]
    ]
)

## Review before Notebook 05

Check warnings, unmatched identifiers and failure records. The next step uses `commercial_bank_ffiec_panel_v2.parquet`.
